# Restricted-input method reference
Source: analysis/travel_pattern_by_neighbourhood/main.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from datetime import datetime, timedelta, time
from typing import Tuple, Dict, Optional

In [ ]:
import os
# Working directory is explicitly configured in the first cell.


In [ ]:
residences['PROP_TYPE'].value_counts()

In [ ]:
park_hulls_hbd

In [ ]:
for park in ["amk", "punggol", "jurong"]:
    print(f"Processing {park} park...")

    residences = gpd.read_file(f"gps_analysis/activities_identification_example/calibrated_residence_{park}_2022_updated.geojson")

    print(residences['PROP_TYPE'].value_counts())

    list_hdb = residences[residences['PROP_TYPE'] == 'HDB']['id'].unique()
    list_landed = residences[residences['PROP_TYPE'] == 'Landed']['id'].unique()
    list_nolanded = residences[(residences['PROP_TYPE'] == 'Non-Landed') | (residences['PROP_TYPE'] == 'EC')]['id'].unique()

    park_hulls = gpd.read_file(f'gis_files/hulls/{park}/breaks_hulls.geojson')
    park_hulls['layer'] = park_hulls['layer'].str.replace('clusters_convex_hulls_', '')
    park_hulls_centroid = park_hulls.to_crs(epsg=3414)
    park_hulls_centroid.drop(columns=['path'], inplace=True)
    # use centroid of hulls to spatially join with segmented hexagons for further analysis
    park_hulls_centroid['geometry'] = park_hulls_centroid.geometry.centroid

    # extract layer from list_hdb
    park_hulls_hbd = park_hulls_centroid[park_hulls_centroid['layer'].isin(list_hdb)]
    park_hulls_landed = park_hulls_centroid[park_hulls_centroid['layer'].isin(list_landed)]
    park_hulls_nolanded = park_hulls_centroid[park_hulls_centroid['layer'].isin(list_nolanded)]

    del park_hulls

    segmented_hex = gpd.read_file(f"gis_files/hexagons/segmented/{park}_hex_agg.geojson")

    segmented_hex = segmented_hex[['grid_id', 'geometry']]

    # SPATIAL JOIN
    segmented_hex_joined = gpd.sjoin(segmented_hex, park_hulls_hbd, how='left', predicate='intersects')

    segmented_hex_joined['start_time'] = pd.to_datetime(segmented_hex_joined['start_time'], errors='coerce')
    segmented_hex_joined['end_time'] = pd.to_datetime(segmented_hex_joined['end_time'], errors='coerce')

    F_h = segmented_hex_joined.groupby('grid_id')['cluster'].nunique()
    duration_sum = segmented_hex_joined.groupby('grid_id')['duration'].sum()


    merged = pd.merge(segmented_hex, F_h.rename('F_h_hdb'), on='grid_id')
    
    merged = pd.merge(merged, duration_sum.rename('D_h_hdb'), on='grid_id')

    segmented_hex = gpd.read_file(f"gis_files/hexagons/segmented/{park}_hex_agg.geojson")

    segmented_hex = segmented_hex[['grid_id', 'geometry']]

    # SPATIAL JOIN
    segmented_hex_joined = gpd.sjoin(segmented_hex, park_hulls_landed, how='left', predicate='intersects')

    segmented_hex_joined['start_time'] = pd.to_datetime(segmented_hex_joined['start_time'], errors='coerce')
    segmented_hex_joined['end_time'] = pd.to_datetime(segmented_hex_joined['end_time'], errors='coerce')

    F_h = segmented_hex_joined.groupby('grid_id')['cluster'].nunique()
    duration_sum = segmented_hex_joined.groupby('grid_id')['duration'].sum()


    merged = pd.merge(merged, F_h.rename('F_h_landed'), on='grid_id')
    
    merged = pd.merge(merged, duration_sum.rename('D_h_landed'), on='grid_id')

    segmented_hex = gpd.read_file(f"gis_files/hexagons/segmented/{park}_hex_agg.geojson")

    segmented_hex = segmented_hex[['grid_id', 'geometry']]

    # SPATIAL JOIN
    segmented_hex_joined = gpd.sjoin(segmented_hex, park_hulls_nolanded, how='left', predicate='intersects')

    segmented_hex_joined['start_time'] = pd.to_datetime(segmented_hex_joined['start_time'], errors='coerce')
    segmented_hex_joined['end_time'] = pd.to_datetime(segmented_hex_joined['end_time'], errors='coerce')

    F_h = segmented_hex_joined.groupby('grid_id')['cluster'].nunique()
    duration_sum = segmented_hex_joined.groupby('grid_id')['duration'].sum()


    merged = pd.merge(merged, F_h.rename('F_h_nolanded'), on='grid_id')
    
    merged = pd.merge(merged, duration_sum.rename('D_h_nolanded'), on='grid_id')
    merged.to_file(f'gps_analysis/general_model/duration_analysis/{park}_duration_analysis_byhousingtypes.geojson', driver='GeoJSON')

In [ ]:
merged